# Databricks RAGアプリチュートリアル - 参考: MLflowによるRAGエージェントの評価

このノートブックは、`2_RAGエージェントの構築.ipynb` で作った **Tool-calling RAG エージェント**を題材に、**MLflow 3 の GenAI 評価機能（`mlflow.genai`）** を本格的に学ぶ *参考* 教材です。上から順に実行するだけで、評価用データセットの自動生成から各種スコアラー・LLM ジャッジによる品質評価までを一通り体験できます。

> 📌 これは既存の `2_RAGエージェントの構築.ipynb`（`mlflow.genai.evaluate` を軽く使うシンプル版）の**発展版**です。エージェントの新規デプロイは行わず、**評価にフォーカス**します。

## このノートブックで学習する内容

1. **MLflow 3 GenAI 評価の全体像と特徴**
2. **プロンプトレジストリ**：システムプロンプトを Unity Catalog でバージョン管理（Beta）
3. **合成データ生成**：文書から評価用テストセットを自動生成
4. **組み込み LLM-as-a-Judge スコアラー**：`RelevanceToQuery` / `Safety` / `Correctness` など
5. **カスタムスコアラー**：`@scorer` 関数とクラスベース `Scorer`
6. **カスタム LLM Judge**：`make_judge` による feedback / trace ベースのジャッジ
7. **統合評価ワークフロー**：`mlflow.genai.evaluate` で全スコアラーをまとめて実行
8. **（任意）本番モニタリング**：スコアラーのサンプリング監視（Beta）

## 実行環境・前提

- **サーバーレスコンピュート**での実行を想定
- `1_PDFのパースとベクトルインデックスの作成.ipynb` を実行済みで、`chunked_documents` テーブルと AI Search Index が作成済み
- Unity Catalog スキーマに対する `CREATE FUNCTION` / `EXECUTE` / `MANAGE` 権限（プロンプトレジストリで使用）

## 参考リンク

- [MLflow 3 GenAI 評価とモニタリング](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/)
- [事前定義ジャッジ・スコアラー](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/scorers/)
- [カスタムジャッジ `make_judge`](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/custom-judge/)
- [プロンプトレジストリ](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/prompt-version-mgmt/prompt-registry/)
- [評価セットの合成](https://learn.microsoft.com/en-us/azure/databricks/agents/agent-evaluation/synthesize-evaluation-set)

## 1. イントロ：MLflow 3 GenAI 評価の全体像と特徴

生成 AI アプリ（RAG エージェントなど）の品質は、従来の ML のような単一の正解ラベルでは測れません。「回答が質問に関連しているか」「検索した文書に基づいているか（幻覚していないか）」「安全か」といった**多面的な観点**を、多くの場合 **LLM 自身をジャッジ（LLM-as-a-Judge）** として使いながら評価します。

MLflow 3 の **`mlflow.genai`** は、この GenAI 評価のために再設計された新しい API 群です。中心となるのは次の4本柱です。

| 柱 | 役割 | 主な API |
|---|---|---|
| **合成データ生成** | 評価用の質問＋期待値を文書から自動作成 | `databricks.agents.evals.generate_evals_df` |
| **スコアラー** | 品質を数値/真偽/フィードバックで測る | `mlflow.genai.scorers`（組み込み & `@scorer` / `Scorer`） |
| **LLM Judge** | 任意の基準を LLM に判定させる | `mlflow.genai.judges.make_judge` |
| **統合評価** | データ×アプリ×スコアラーをまとめて実行 | `mlflow.genai.evaluate` |
| **プロンプト管理** | プロンプトを UC でバージョン管理 | `mlflow.genai.register_prompt` / `load_prompt` |

### MLflow 3 GenAI 評価の特徴

`mlflow.genai` は、次のような特徴を持つように設計されています。

| 特徴 | 内容 |
|---|---|
| **トレースネイティブ** | 評価が **MLflow Tracing** 前提。`predict_fn` を1回呼ぶと1本のトレースが作られ、スコアラーはそのトレース（入力・出力・検索スパンなど）を直接参照して採点します |
| **LLM-as-a-Judge 中心** | 組み込みジャッジに加え、`make_judge` で自然言語の指示から独自ジャッジを作成可能。人手フィードバックにジャッジを近づける整合（alignment）にも対応 |
| **スコアラーの柔軟性** | 組み込みスコアラー・`@scorer` 関数・クラスベース `Scorer` を、同じ `evaluate` に混在させて実行できます |
| **オフライン評価と本番監視の一貫性** | 開発時のオフライン評価で使った `@scorer` を、そのまま本番トレースのサンプリング監視にも流用できます |
| **Unity Catalog 統合** | プロンプト・評価データセット・評価結果を UC のガバナンス（権限・リネージ）の下で管理できます |
| **構造化された評価データ** | 入力は `inputs`、正解は `expectations.*` としてネスト構造で扱い、合成データ生成や `EvaluationDataset` がスキーマを保証します |

> 💡 中でも重要なのが **トレースとの統合**です。`predict_fn` 1回＝トレース1本という対応により、検索の根拠性を測る `RetrievalGroundedness` などは**トレース内の RETRIEVER スパン**を参照して採点します（手順9・手順10で実際に確認します）。

> 📎 なお、評価 API は MLflow 2 の `mlflow.evaluate()` から `mlflow.genai.evaluate()` に刷新され、現在は **MLflow 3 系が推奨**です（2 系はソフト非推奨）。本ノートブックは MLflow 3 系のみを扱います。

## 2. ライブラリの準備

評価に使うライブラリをインストールします。`make_judge`（カスタム LLM ジャッジ）は **`mlflow >= 3.4.0`**、合成データ生成は **`databricks-agents`** が必要です。評価対象の RAG エージェントを組むために `databricks-langchain` / `langgraph` も入れます。

> 💡 `%pip install` の後に `dbutils.library.restartPython()` を呼ぶと Python が再起動し、**それ以前に定義した変数はすべて消えます**。そのため widget やパラメータの定義は必ずこのセルより後（手順3以降）で行います。

In [ ]:
# 評価・合成データ生成・エージェント構築用ライブラリ
%pip install -U -qqqq "mlflow[databricks]>=3.4.0" "databricks-agents>=1.9.3" "databricks-langchain>=0.17.0" "langgraph>=1.1.0"
dbutils.library.restartPython()

## 3. パラメータ設定（widget）

環境依存の値を widget で指定します。`1_...` ノートブックと同じカタログ／スキーマ／インデックス名を指定してください。

- **`JUDGE_MODEL_ENDPOINT`**：`make_judge` のカスタムジャッジが使う LLM エンドポイント。`databricks-claude-sonnet-4-5` などの基盤モデルを指定します（`make_judge` には `databricks:/<エンドポイント名>` の形式で渡します）。

In [ ]:
# パラメータ設定（widget から取得） — restartPython の後に定義すること
dbutils.widgets.text("CATALOG_NAME", "skato", "カタログ名")
dbutils.widgets.text("SCHEMA_NAME", "rag_workshop", "スキーマ名")
dbutils.widgets.text("VECTOR_INDEX_NAME", "chunked_document_vs_index", "AI Search Index名")
dbutils.widgets.text("SOURCE_TABLE_NAME", "chunked_documents", "合成データ生成の元テーブル")
dbutils.widgets.text("LLM_ENDPOINT_NAME", "databricks-claude-sonnet-4-5", "エージェントのLLMエンドポイント")
dbutils.widgets.text("JUDGE_MODEL_ENDPOINT", "databricks-claude-sonnet-4-5", "カスタムJudge用LLMエンドポイント")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
VECTOR_INDEX_NAME = dbutils.widgets.get("VECTOR_INDEX_NAME")
SOURCE_TABLE_NAME = dbutils.widgets.get("SOURCE_TABLE_NAME")
LLM_ENDPOINT_NAME = dbutils.widgets.get("LLM_ENDPOINT_NAME")
JUDGE_MODEL_ENDPOINT = dbutils.widgets.get("JUDGE_MODEL_ENDPOINT")

# 完全修飾名
VS_INDEX_FULLNAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{VECTOR_INDEX_NAME}"
SOURCE_TABLE_FULLNAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{SOURCE_TABLE_NAME}"
# make_judge に渡すモデル URI（provider:/endpoint 形式）
JUDGE_MODEL_URI = f"databricks:/{JUDGE_MODEL_ENDPOINT}"

print(f"AI Search Index      : {VS_INDEX_FULLNAME}")
print(f"合成データ元テーブル : {SOURCE_TABLE_FULLNAME}")
print(f"エージェントLLM      : {LLM_ENDPOINT_NAME}")
print(f"Judge モデル URI     : {JUDGE_MODEL_URI}")

## 4. プロンプトレジストリ（Beta）

**プロンプトレジストリ**は、システムプロンプトなどのプロンプト文字列を **Unity Catalog 上でバージョン管理**する機能です。コードとプロンプトを分離でき、どのプロンプトのどのバージョンで評価したかを追跡できます。

- **登録**：`mlflow.genai.register_prompt(name=..., template=..., commit_message=..., tags=...)`
- **読み込み**：`mlflow.genai.load_prompt(name_or_uri="prompts:/<name>/<version>")`
- **テンプレート変数**：`{{ 変数名 }}`（二重波括弧）。`.format(変数名=...)` で埋め込みます。
- **命名**：Databricks では UC の**3階層名**（`catalog.schema.name`）を使い、URI は `prompts:/catalog.schema.name/1` の形式になります。

> ⚠️ **ステータス: Beta**。ワークスペース管理者が [Previews] ページで有効化している必要があります。対象スキーマに `CREATE FUNCTION` / `EXECUTE` / `MANAGE` 権限が必要です。

ここでは RAG エージェント用のシステムプロンプトを登録し、`{{ domain }}` を埋め込んで読み込みます。読み込んだプロンプトは手順9のエージェント構築でそのまま使います。

In [ ]:
import mlflow

PROMPT_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.rag_system_prompt"

# システムプロンプトをテンプレートとして登録（{{ domain }} が変数）
prompt = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=(
        "あなたは{{ domain }}に関する専門的なアシスタントです。"
        "登録された AI Search Index を活用し、検索された文書の内容に基づいて、"
        "日本語で丁寧かつ正確に回答してください。不確かな情報は推測しないでください。"
    ),
    commit_message="初版: RAGエージェント用システムプロンプト",
    tags={"task": "rag", "language": "ja"},
)

print(f"登録しました: name={prompt.name}, version={prompt.version}")
print(f"URI         : {prompt.uri}")
print(f"テンプレート : {prompt.template}")

In [ ]:
# 登録したプロンプトを読み込み、変数 {{ domain }} を埋め込んで最終文字列を得る
loaded_prompt = mlflow.genai.load_prompt(name_or_uri=f"prompts:/{PROMPT_NAME}/{prompt.version}")

# .format(...) でテンプレート変数を置換
SYSTEM_PROMPT = loaded_prompt.format(domain="生成AI開発")

print("=== 評価対象エージェントに渡すシステムプロンプト ===")
print(SYSTEM_PROMPT)

## 5. 合成データ生成（評価用テストセットの作成）

評価には「質問」と「期待される事実（ground truth）」のセットが必要ですが、人手で用意するのは大変です。**`generate_evals_df`** を使うと、文書から評価用テストセットを **自動生成** できます。

- 入力 `docs`：**`content`**（文書本文）と **`doc_uri`**（文書URI）の2列を持つ pandas / Spark DataFrame
- 主な引数：`num_evals`（生成する評価件数）、`agent_description`（エージェントの説明）、`question_guidelines`（質問生成の方針）
- 出力（MLflow 3）：`inputs`（`{"messages": [...]}` 形式）、`expectations.expected_facts`（期待事実）、`expectations.expected_retrieved_context`（生成元の文脈）

`1_...` で作成した `chunked_documents` テーブルの本文（`chunk_content`）を文書として使います。

> ⚠️ サーバーレス egress control が有効なワークスペースでは合成データ生成は利用できません。件数（`num_evals`）は LLM 呼び出しを伴うため、ハンズオンでは少なめ（5件）にしています。

In [ ]:
# chunked_documents から content / doc_uri の2列を用意（ハンズオン用に少数へ）
docs_df = (
    spark.table(SOURCE_TABLE_FULLNAME)
    .selectExpr("chunk_content AS content", "path AS doc_uri")
    .where("length(chunk_content) > 100")  # 短すぎるチャンクは除外
    .limit(15)
    .toPandas()
)

print(f"合成データの元文書: {len(docs_df)} 件")
display(docs_df.head())

In [ ]:
from databricks.agents.evals import generate_evals_df

agent_description = (
    "生成AI開発（GenAI開発ワークフロー、MLOps、エージェント設計など）に関する"
    "技術文書を検索して日本語で回答する RAG エージェント。"
)
question_guidelines = """
# 想定ユーザー
- Databricks で生成AIアプリを開発し始めたエンジニア

# 質問の例
- RAGの評価にはどんな指標を使えばよいですか？
- エージェントの品質をどうモニタリングしますか？

# 追加のガイドライン
- 質問は簡潔で、人間が実際に尋ねそうな自然な日本語にすること
"""

# 評価用テストセットを自動生成
evals = generate_evals_df(
    docs_df,
    num_evals=5,
    agent_description=agent_description,
    question_guidelines=question_guidelines,
)

print(f"生成された評価件数: {len(evals)}")
print("列:", list(evals.columns))
display(evals)

## 6. 組み込み LLM-as-a-Judge スコアラー

`mlflow.genai.scorers` には、代表的な品質観点を LLM ジャッジで採点する **組み込みスコアラー**が用意されています。すべて `model=`（`"databricks:/<endpoint>"` 形式）を任意で指定でき、省略時は Databricks 標準のジャッジモデルが使われます。呼び出すと `value`（多くは `"yes"`/`"no"` や真偽）と `rationale`（判定理由）を持つ `Feedback` が返ります。

| スコアラー | 測る観点 | 必要なデータ | トレース要否 |
|---|---|---|---|
| `RelevanceToQuery` | 回答が質問に関連しているか | `inputs`, `outputs` | 不要 |
| `Safety` | 回答が安全か（有害でないか） | `outputs` | 不要 |
| `Correctness` | 期待事実に照らして正しいか | `inputs`, `outputs`, **`expectations`** | 不要 |
| `Guidelines` | 与えたガイドラインを満たすか | `inputs`, `outputs`（＋`guidelines`） | 不要 |
| `RetrievalGroundedness` | 回答が検索文書に基づくか（幻覚検知） | 検索文脈＋回答 | **必要（RETRIEVER スパン）** |
| `RetrievalRelevance` | 検索された文書が質問に関連するか | 検索文書＋質問 | **必要（RETRIEVER スパン）** |

> 💡 `RetrievalGroundedness` / `RetrievalRelevance` は、トレース内に `span_type="RETRIEVER"` のスパンを必要とします。手順9で `VectorSearchRetrieverTool` + `mlflow.langchain.autolog()` を使うことで自動的にこのスパンが記録され、統合評価（手順10）で採点できるようになります。
>
> ⚠️ 補足: `RelevanceToContext` というスコアラーは**存在しません**。検索文脈の妥当性は `RetrievalRelevance`（または `RetrievalSufficiency`）を使います。

まずはトレース不要のスコアラーを、サンプルの入出力に対して直接呼び出して挙動を確認します。

In [ ]:
from mlflow.genai.scorers import RelevanceToQuery, Safety, Correctness, Guidelines

# サンプルの入力・出力（実際の評価では predict_fn の出力が使われる）
sample_input = {"request": "MLflowのプロンプトレジストリの特徴は？"}
sample_output = {"response": "MLflowのプロンプトレジストリは、プロンプトをUnity Catalog上でバージョン管理できる機能です。"}

# スコアラーごとに必要な引数が異なる点に注意（下の表を参照）
# 質問との関連性（inputs と outputs が必要）
fb = RelevanceToQuery()(inputs=sample_input, outputs=sample_output)
print(f"[relevance_to_query] value={fb.value}\n  理由: {fb.rationale}\n")

# 安全性（outputs のみで採点。inputs を渡すとエラーになる）
fb = Safety()(outputs=sample_output)
print(f"[safety] value={fb.value}\n  理由: {fb.rationale}\n")

# 正確性（inputs / outputs に加え、期待事実 expectations が必要）
fb = Correctness()(
    inputs=sample_input,
    outputs=sample_output,
    expectations={"expected_facts": ["プロンプトをバージョン管理できる", "Unity Catalogと統合されている"]},
)
print(f"[correctness] value={fb.value}\n  理由: {fb.rationale}\n")

# ガイドライン遵守（name と guidelines が必須。回答=response, 質問=request として参照される）
japanese_guideline = Guidelines(name="japanese_only", guidelines=["回答は必ず日本語で書かれていること"])
fb = japanese_guideline(inputs=sample_input, outputs=sample_output)
print(f"[japanese_only] value={fb.value}\n  理由: {fb.rationale}")

## 7. カスタムスコアラー

組み込みスコアラーで足りない独自の観点は、**カスタムスコアラー**で定義できます。2つの書き方があります。

### 7-1. `@scorer` デコレータ（関数ベース）

```python
@scorer(name=..., aggregations=[...])
def my_metric(inputs, outputs, expectations, trace): ...
```

- 引数 `inputs` / `outputs` / `expectations` / `trace` は**名前で自動注入**されます（必要なものだけ宣言）。
- 戻り値は `int` / `float` / `bool` / `str` / `Feedback` / `list[Feedback]`。
- `aggregations` に指定できる集計は **`"min"`, `"max"`, `"mean"`, `"median"`, `"variance"`, `"p90"`**（`p99` は不可）。

### 7-2. クラスベース `Scorer`

`Scorer`（Pydantic ベース）を継承し、`__call__` をオーバーライドします。`name` フィールドは必須です。

> ⚠️ クラスベース `Scorer` は**本番モニタリングでは非対応**です（手順11）。モニタリングには `@scorer` 関数を使います。

In [ ]:
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback


@scorer(aggregations=["mean", "min", "max", "p90"])
def response_length(outputs) -> int:
    """応答の文字数を返すカスタムメトリクス（集計: mean/min/max/p90）。"""
    text = outputs.get("response", "") if isinstance(outputs, dict) else str(outputs)
    return len(text)


@scorer
def has_source_reference(outputs) -> Feedback:
    """回答が根拠（文書/検索）に言及していそうかを簡易チェックし、理由付きで返す。"""
    text = outputs.get("response", "") if isinstance(outputs, dict) else str(outputs)
    cited = any(k in text for k in ["文書", "ドキュメント", "検索", "によると", "参照"])
    return Feedback(
        value=cited,
        rationale="根拠への言及が見られる" if cited else "根拠への明示的な言及が見られない",
    )


# 直接呼び出しで挙動確認（戻り値が Feedback の場合は .value を表示）
_r = response_length(outputs={"response": "これはテスト用の応答です。"})
print("response_length:", getattr(_r, "value", _r))
_r2 = has_source_reference(outputs={"response": "検索した文書によると、RAGは有効です。"})
print("has_source_reference:", _r2.value, "/", _r2.rationale)

In [ ]:
import re
from mlflow.genai.scorers import Scorer
from mlflow.entities import Feedback


class JapaneseResponseScorer(Scorer):
    """回答に日本語（かな/カナ/漢字）が含まれるかを判定するクラスベース Scorer。"""

    name: str = "japanese_response"

    def __call__(self, outputs) -> Feedback:
        text = outputs.get("response", "") if isinstance(outputs, dict) else str(outputs or "")
        has_jp = bool(re.search(r"[ぁ-んァ-ヶ一-龥]", text))
        return Feedback(
            value=has_jp,
            rationale="日本語（かな/カナ/漢字）を含む" if has_jp else "日本語を含まない",
        )


# 挙動確認
jp_scorer = JapaneseResponseScorer()
print(jp_scorer(outputs={"response": "これは日本語の応答です。"}).value)
print(jp_scorer(outputs={"response": "This is English only."}).value)

## 8. カスタム LLM Judge（`make_judge`）

`make_judge`（`mlflow >= 3.4.0`）は、**自然言語の指示だけで独自の LLM ジャッジ**を作れる機能です。組み込みスコアラーにない任意の観点を、プロンプトで柔軟に定義できます。

```python
judge = make_judge(name=..., instructions="... {{ inputs }} {{ outputs }} ...",
                   feedback_value_type=bool, model="databricks:/...")
```

- **指示文で使えるテンプレート変数は4種のみ**：`{{ inputs }}` / `{{ outputs }}` / `{{ expectations }}` / `{{ trace }}`（最低1つは必須）。
- **判定の型 `feedback_value_type`**：`bool` / `int` / `float` / `str` / `Literal[...]` など（省略時は `str`）。
- **feedback ジャッジ vs trace ベースジャッジ**：
  - 指示文が `{{ inputs }}`/`{{ outputs }}`/`{{ expectations }}` を参照 → **feedback ジャッジ**。`model` は任意（省略時は Databricks 標準ジャッジ）。
  - 指示文が **`{{ trace }}`** を参照 → **trace ベースジャッジ**（トレース全体を分析）。この場合 **`model` の指定が必須**。

ここでは2種類作ります。統合評価（手順10）で `scorers` に渡して使います。

In [ ]:
from mlflow.genai.judges import make_judge

# (1) feedback ジャッジ: 回答が簡潔かを bool で判定（trace 不参照なので model 省略可）
conciseness_judge = make_judge(
    name="conciseness",
    instructions=(
        "ユーザーの質問 {{ inputs }} に対して、回答 {{ outputs }} は"
        "冗長でなく簡潔に要点を答えていますか？ true または false で判定してください。"
    ),
    feedback_value_type=bool,
)

# 直接呼び出しで挙動確認（feedback ジャッジは inputs/outputs だけで動く）
fb = conciseness_judge(
    inputs={"messages": [{"role": "user", "content": "プロンプトレジストリとは？"}]},
    outputs={"response": "プロンプトをUnity Catalogでバージョン管理する機能です。"},
)
print(f"conciseness: value={fb.value}")
print(f"  理由: {fb.rationale}")

In [ ]:
# (2) trace ベースジャッジ: 実行トレースを分析し、検索ツールを使ったかを判定
#     {{ trace }} を参照するため model の指定が必須
retrieval_tool_judge = make_judge(
    name="retrieval_tool_used",
    instructions=(
        "実行トレース {{ trace }} を分析してください。エージェントが最終回答の前に"
        "検索ツール（retriever）を呼び出して文書を取得していれば true、"
        "取得していなければ false と判定してください。"
    ),
    feedback_value_type=bool,
    model=JUDGE_MODEL_URI,  # trace ベースジャッジでは必須
)

print("trace ベースジャッジを定義しました:", retrieval_tool_judge.name)
print("（トレースが必要なため、実際の採点は手順10の統合評価で行います）")

## 9. 評価対象エージェントの構築（`predict_fn`）

統合評価では、評価対象のアプリを **`predict_fn`** として渡します。`generate_evals_df` が生成した `inputs`（`{"messages": [...]}`）が `predict_fn` にキーワード引数として展開されるため、ここでは `messages` を受け取る関数を定義します。

`2_RAGエージェントの構築.ipynb` と同じ構成（`create_agent` + `VectorSearchRetrieverTool`）で LangGraph エージェントを組み、**手順4でプロンプトレジストリから読み込んだ `SYSTEM_PROMPT`** をそのまま使います。

**ポイント（トレース連携）:**
- `@mlflow.trace` で `predict_fn` 全体が1本のトレースになります。
- `mlflow.langchain.autolog()` により、`VectorSearchRetrieverTool` の検索が **RETRIEVER スパン**として自動記録されます。これで `RetrievalGroundedness` / `RetrievalRelevance` / trace ベースジャッジが採点可能になります。
- `predict_fn` は **JSON シリアライズ可能な dict** を返す必要があります（ここでは `{"response": ...}`）。

In [ ]:
import mlflow
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langchain.agents import create_agent

# LangChain / LangGraph の自動トレースを有効化（RETRIEVER スパンが記録される）
mlflow.langchain.autolog()

_retriever_tool = VectorSearchRetrieverTool(
    index_name=VS_INDEX_FULLNAME,
    tool_description=(
        "生成AI開発に関する技術文書やベストプラクティスを検索するツール。"
        "GenAI開発ワークフロー、MLOps、エージェント設計などの質問に使用する。"
    ),
)

# 手順4でプロンプトレジストリから読み込んだ SYSTEM_PROMPT を使用
_rag_agent = create_agent(
    model=ChatDatabricks(endpoint=LLM_ENDPOINT_NAME),
    tools=[_retriever_tool],
    system_prompt=SYSTEM_PROMPT,
)


@mlflow.trace
def rag_predict_fn(messages):
    """評価対象の RAG エージェント。inputs={'messages': [...]} を受け取り {'response': ...} を返す。"""
    result = _rag_agent.invoke({"messages": messages})
    return {"response": result["messages"][-1].content}


# 動作確認（1件だけ呼び出してトレースが作られることを確認）
_demo = rag_predict_fn(messages=[{"role": "user", "content": "生成AIの開発ワークフローについて教えてください"}])
print(_demo["response"][:300])

## 10. 統合評価ワークフロー（`mlflow.genai.evaluate`）

いよいよ全スコアラーをまとめて実行します。**`mlflow.genai.evaluate`** に「評価データ（手順5）」「評価対象アプリ（手順9の `predict_fn`）」「スコアラー一式（組み込み＋カスタム＋LLM ジャッジ）」を渡します。

MLflow は各データ行に対して `predict_fn` を実行してトレースを取得し、各スコアラーで採点します。結果は現在の MLflow エクスペリメントに **Evaluation Run** として記録され、MLflow UI で行ごと・スコアラーごとに比較できます。

> ⚠️ このセルは評価件数ぶん LLM（エージェント本体＋各ジャッジ）を呼び出すため、数分かかることがあります。

In [ ]:
from mlflow.genai.scorers import (
    RelevanceToQuery,
    Safety,
    Correctness,
    RetrievalGroundedness,
    RetrievalRelevance,
)

results = mlflow.genai.evaluate(
    data=evals,               # 手順5で生成した評価データ
    predict_fn=rag_predict_fn,  # 手順9の評価対象エージェント
    scorers=[
        # --- 組み込み（トレース不要） ---
        RelevanceToQuery(),
        Safety(),
        Correctness(),
        # --- 組み込み（RETRIEVER スパンを含むトレースが必要） ---
        RetrievalGroundedness(),
        RetrievalRelevance(),
        # --- カスタムスコアラー（手順7） ---
        response_length,
        has_source_reference,
        JapaneseResponseScorer(),
        # --- カスタム LLM ジャッジ（手順8） ---
        conciseness_judge,
        retrieval_tool_judge,
    ],
)

# 集計スコアの表示（属性名はバージョン差異があるため防御的に）
metrics = getattr(results, "metrics", None)
if metrics:
    print("=== 集計スコア ===")
    for k, v in metrics.items():
        print(f"{k}: {v}")
print("\n詳細は右サイドバーの [Experiments] から対象 Run を開き、[Traces] / [Evaluations] タブで")
print("行ごと・スコアラーごとの結果を確認できます（プロンプトのバージョン間比較などにも活用できます）。")

## 11. （任意）本番モニタリング（Beta）

本番運用では、ライブトラフィックのトレースに対してスコアラーを**サンプリング実行**し、品質を継続監視できます。オフライン評価との違いは、`@scorer` 関数（または組み込みジャッジ）を **`.register()` → `.start()`** で登録・起動する点です。

- `ScorerSamplingConfig(sample_rate=..., filter_string=...)` でサンプリング率と対象トレースを指定
- `list_scorers()` / `get_scorer(name=...)` / `delete_scorer(name=...)` で管理、`.stop()` で停止
- **クラスベース `Scorer` は非対応**。監視用 `@scorer` 関数は self-contained（import を関数内に書く）である必要があります。

> ⚠️ **ステータス: Beta**。ワークスペースでプレビュー有効化が必要で、監視対象となる本番トレース（デプロイ済みアプリ）が前提です。本ノートブックではデプロイを行わないため、以下は**設定方法の例示**として、登録直後に停止・削除してクリーンアップします。

In [ ]:
from mlflow.genai.scorers import scorer, ScorerSamplingConfig, list_scorers, delete_scorer


@scorer
def response_non_empty(outputs):
    # 監視用スコアラーは self-contained にする（必要な import は関数内に書く）
    text = outputs.get("response", "") if isinstance(outputs, dict) else str(outputs)
    return len(text.strip()) > 0


MONITOR_NAME = "response_non_empty_monitor"
try:
    # 登録 → サンプリング監視を開始（スコアラーはイミュータブルなので都度代入し直す）
    monitor = response_non_empty.register(name=MONITOR_NAME)
    monitor = monitor.start(sampling_config=ScorerSamplingConfig(sample_rate=0.5))
    print("登録済みスコアラー:", [s.name for s in list_scorers()])

    # ハンズオンでは監視を残さないよう停止・削除
    monitor = monitor.stop()
    delete_scorer(name=MONITOR_NAME)
    print(f"'{MONITOR_NAME}' を停止・削除しました（クリーンアップ完了）")
except Exception as e:
    print("本番モニタリングはプレビュー有効化とトレース対象が必要です。スキップします:")
    print(" ", e)

## 12. まとめと次のステップ

### このノートブックで学んだこと

- **MLflow 3 GenAI 評価の全体像**と、MLflow 2 の `mlflow.evaluate` からの違い
- **プロンプトレジストリ**：システムプロンプトを UC でバージョン管理し、エージェントから参照（Beta）
- **合成データ生成**：`generate_evals_df` で文書から評価テストセットを自動生成
- **組み込みスコアラー**：`RelevanceToQuery` / `Safety` / `Correctness` / `RetrievalGroundedness` / `RetrievalRelevance`
- **カスタムスコアラー**：`@scorer` 関数とクラスベース `Scorer`
- **カスタム LLM ジャッジ**：`make_judge` の feedback ジャッジ / trace ベースジャッジ
- **統合評価**：`mlflow.genai.evaluate` で全スコアラーをまとめて実行し、MLflow UI で比較
- **（任意）本番モニタリング**：`@scorer` + `.register()` / `.start()` のサンプリング監視（Beta）

### 各機能のステータス（2026年時点）

| 機能 | ステータス |
|---|---|
| `mlflow.genai.evaluate` / 組み込みスコアラー | GA |
| `make_judge`（カスタムジャッジ） | 一般提供（`mlflow >= 3.4.0`） |
| 合成データ生成 `generate_evals_df` | GA |
| プロンプトレジストリ | **Beta** |
| 本番モニタリング（スコアラーのサンプリング監視） | **Beta** |

### 次のステップ

1. **ジャッジの整合（alignment）**：`judge.align(traces)` で人手フィードバックにジャッジを近づける（`mlflow >= 3.4.0`）
2. **プロンプトのバージョン比較**：プロンプトを更新して version 2 を登録し、評価スコアを version 間で比較
3. **本番モニタリング**：エージェントをデプロイ（`2_RAGエージェントの構築.ipynb`）した上で、スコアラーのサンプリング監視を有効化

### 参考リンク

- [MLflow 3 GenAI 評価とモニタリング](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/)
- [事前定義ジャッジ・スコアラー](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/scorers/)
- [カスタムスコアラー](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/custom-scorers)
- [カスタムジャッジ `make_judge`](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/custom-judge/)
- [ジャッジの整合（alignment）](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/align-judges)
- [プロンプトレジストリ](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/prompt-version-mgmt/prompt-registry/)